# LaTeX table workshop

One notebook for paper tables. It reads portable report CSVs, exports selected columns, and preserves unrounded values and relative-path provenance. Add `\usepackage{booktabs}` to your paper. Bold marks the primary metric, not significance.

In [1]:
from pathlib import Path
import os, sys
import pandas as pd
from IPython.display import display, Image
ROOT = Path.cwd() if (Path.cwd()/"src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.report_bundle import read_summary, verify_bundle
from src.utils import focus_table, display_details
REPORTS = Path(os.environ.get("SAERT_REPORTS_DIR", ROOT/"reports"))
verify_bundle(REPORTS)
def load(section, name): return read_summary(section, name, REPORTS)
def figure(section, name): display(Image(filename=str(REPORTS/section/(name+".png"))))
from src.table_export import export_table
OUTPUT = REPORTS/"tables"


In [2]:
catalog = {
 "sae_prediction":dict(section="sae_prediction",name="primary_results",columns=["Corpus","State","SAE gain (pp)","SAE 95% interval"],metric="SAE gain (pp)",caption="SAE gains beyond lexical controls and matched-context surprisal."),
 "sae_dense_contrast":dict(section="sae_prediction",name="primary_results",columns=["Corpus","State","SAE minus dense (pp)","Difference 95% interval"],metric="SAE minus dense (pp)",caption="Paired SAE-versus-dense comparison. Positive values favor SAE."),
 "scaling_sensitivity":dict(section="scaling_sensitivity",name="comparison",columns=["corpus","state","floor_gain_pp","floor_ci_low_pp","floor_ci_high_pp"],metric="floor_gain_pp",caption="Exploratory fixed-L01 sensitivity using a training-derived scale floor.")}
TABLES = list(catalog)  # Choose one or more names.
for name in TABLES:
    spec = catalog[name]; frame=load(spec["section"],spec["name"])
    export_table(frame,OUTPUT/(name+".tex"),columns=spec["columns"],caption=spec["caption"],label="tab:"+name.replace("_","-"),
      decimals={c:2 for c in spec["columns"] if pd.api.types.is_numeric_dtype(frame[c])},signed=[spec["metric"]],emphasize=[spec["metric"]],wide=True,
      notes="Gains in R2 percentage points. Intervals resample texts conditional on saved predictions, excluding participant and refitting uncertainty.",
      sources=[REPORTS/spec["section"]/(spec["name"]+".csv")])
    display_details(name,frame[spec["columns"]])
print("Tables saved under reports/tables")

Corpus,State,SAE gain (pp),SAE 95% interval
Provo · FFD,Before word,2.869,"[+1.77, +3.98]"
Provo · FFD,After word,6.576,"[+5.42, +7.78]"
Natural Stories · SPR,Before word,0.820,"[+0.19, +1.37]"
Natural Stories · SPR,After word,3.840,"[+2.35, +5.18]"


Corpus,State,SAE minus dense (pp),Difference 95% interval
Provo · FFD,Before word,1.801,"[+0.67, +2.88]"
Provo · FFD,After word,3.779,"[+2.73, +4.88]"
Natural Stories · SPR,Before word,-1.786,"[-3.49, +0.25]"
Natural Stories · SPR,After word,-2.392,"[-4.02, -0.45]"


corpus,state,floor_gain_pp,floor_ci_low_pp,floor_ci_high_pp
provo,prefix,3.412,2.533,4.333
provo,post,6.531,5.277,7.819
natural_stories,prefix,0.617,0.200,0.949
natural_stories,post,2.128,1.469,2.814


Tables saved under reports/tables


## Any other table

Use the same exporter with your DataFrame. Specify useful columns, precision, units and uncertainty. The example is disabled until you provide a CSV path.

In [3]:
CUSTOM_CSV = None  # Example: REPORTS/"baseline/model_comparison.csv"
if CUSTOM_CSV is not None:
    custom = pd.read_csv(CUSTOM_CSV)
    export_table(custom,OUTPUT/"custom_table.tex",caption="Replace with your caption",label="tab:custom",sources=[CUSTOM_CSV])